<h2 align='center'>Codebasics ML Course: ML Flow Tutorial</h2>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier 
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# step 1 : Create an imbalance binary classification report 
X,y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, weights=[0.9,0.1], flip_y=0, random_state=42)
np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100], dtype=int64))

In [3]:
# split the dataset into training and testing sets
X_train ,X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, stratify=y, random_state=42)


## Handle imbalance class

In [4]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619], dtype=int64))

### Track Experiments

In [5]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [6]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [9]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost


In [10]:
# Dagshub setup
import dagshub
dagshub.init(repo_owner='reachmurali2', repo_name='mlflow_dagshub_demo', mlflow=True)

Initialized MLflow to track repo "reachmurali2/mlflow_dagshub_demo"

Repository reachmurali2/mlflow_dagshub_demo initialized!

In [11]:
# Ideally you will not require following 4 lines if you have started and do not have any previous dagshub credentials on your computer
import os
os.environ['MLFLOW_TRACKING_USERNAME']='reachmurali2'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '15c692c817fb35b4abe13a93649923cf3557dd82'
os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow'

# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
# mlflow.set_tracking_uri("http://localhost:5000")

# mlflow.set_tracking_uri("https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2025/04/30 13:32:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Logistic Regression at: https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/bb3ba62fb4674f39a03199aeb8e03e12
🧪 View experiment at: https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow/#/experiments/0


2025/04/30 13:33:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/c404f96cb86a49b9b9f34303d19543e4
🧪 View experiment at: https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow/#/experiments/0


2025/04/30 13:33:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier at: https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/f71cdb2bb77b4637984af3f834a7601f
🧪 View experiment at: https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow/#/experiments/0


2025/04/30 13:33:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier With SMOTE at: https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow/#/experiments/0/runs/6e96c8b214a449b89e1f09409dddb727
🧪 View experiment at: https://dagshub.com/reachmurali2/mlflow_dagshub_demo.mlflow/#/experiments/0


## Register the Model

In [12]:
model_name = 'XGB-Smote'
run_id = input('Please type RunID')
model_uri = f'runs:/{run_id}/model_name'

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri = model_uri, name=model_name)

Please type RunID 93aa37041ca9463398a7edbc1eb7a4d0


Successfully registered model 'XGB-Smote'.
2025/03/08 12:41:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1


🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/887408098861223224/runs/93aa37041ca9463398a7edbc1eb7a4d0
🧪 View experiment at: http://localhost:5000/#/experiments/887408098861223224


Created version '1' of model 'XGB-Smote'.


# Updated Register Model 

In [17]:
model_name = "XGB-Smote"
run_id=input('Enter Run ID:')
model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(
    model_uri, model_name
)

Enter Run ID: 93aa37041ca9463398a7edbc1eb7a4d0


Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2025/03/08 12:48:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


In [14]:
# Load the Model

In [18]:
model_version = 1
moel_url = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

# Transition the Model to Production

In [23]:
current_model_uri = f"models:/{model_name}@challenger"
production_model_name = "anomaly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1741418763754, current_stage='None', description='', last_updated_timestamp=1741418763754, name='anomaly-detection-prod', run_id='93aa37041ca9463398a7edbc1eb7a4d0', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [24]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

Please refer to following to learn more about model registry

https://mlflow.org/docs/latest/model-registry.html#model-registry-workflows to learn 